# TP2 - Compresión de imágenes sin pérdida con Huffman
Inferencia y Estimación - Primavera 2026

## 0. Base común

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

from funciones_auxiliares_tp2 import huffman_code, huffman_decode

IMG_DIR = 'Archivos TP2-20261023'   
IMAGENES = ['img_01.jpg', 'img_02.jpg', 'img_03.jpg', 'img_04.jpg', 'img_05.jpg']

In [ ]:
def cargar_imagen(nombre):
    """Carga la imagen en escala de grises como matriz int (0-255)."""
    img = Image.open(f'{IMG_DIR}/{nombre}').convert('L')
    return np.array(img, dtype=np.int16)   # int16 para que los residuos no hagan overflow


def pmf(simbolos):
    """PMF empírica: dict {simbolo: probabilidad} a partir de un array de símbolos."""
    simbolos = np.asarray(simbolos).ravel()
    cuentas = Counter(simbolos.tolist())
    n = simbolos.size
    return {s: c / n for s, c in cuentas.items()}


def entropia(p):
    """Entropía en bits de una PMF (dict o array). Ignora probabilidades nulas."""
    v = np.array(list(p.values()) if isinstance(p, dict) else p, dtype=float)
    v = v[v > 0]
    return float(-np.sum(v * np.log2(v)))


def longitud_media(p, codigo):
    """L = sum p(x) l(x): longitud media teórica del código."""
    return sum(p[s] * len(codigo[s]) for s in p)


def codificar(simbolos, codigo):
    """Codifica una secuencia de símbolos a un string de bits."""
    return ''.join(codigo[s] for s in np.asarray(simbolos).ravel().tolist())


def bits_promedio(bits, n_simbolos):
    """Bits promedio reales por símbolo de la secuencia codificada."""
    return len(bits) / n_simbolos

In [ ]:
def residuos_filas(img):
    """Residuos por fila: y[i,j] = x[i,j] - x[i,j-1], j=2..N.
    Retorna (primera_columna, residuos), residuos de forma (M, N-1).
    No se diferencia entre el fin de una fila y el inicio de la siguiente."""
    return img[:, 0].copy(), np.diff(img, axis=1)


def reconstruir_filas(primera_col, res):
    """Inversa de residuos_filas."""
    primera_col = np.asarray(primera_col)[:, None]
    return np.concatenate([primera_col, primera_col + np.cumsum(res, axis=1)], axis=1)


def residuos_serpentina(img):
    """Recorrido en serpentina (filas impares de derecha a izquierda).
    Retorna (x_inicial, residuos) con residuos de forma (M, N-1) en orden de recorrido.
    Un solo píxel inicial para toda la imagen."""
    s = img.copy()
    s[1::2] = s[1::2, ::-1]
    # Los residuos entre filas se calculan entre píxeles vecinos (último de una fila y
    # el de abajo, que en serpentina es el primero de la siguiente), por eso es válido
    # aplanar el recorrido.
    flat = s.ravel()
    return int(flat[0]), np.diff(flat)


def reconstruir_serpentina(x0, res, shape):
    flat = np.concatenate([[x0], x0 + np.cumsum(res)])
    s = flat.reshape(shape)
    s[1::2] = s[1::2, ::-1]
    return s


def pares_vecinos(img):
    """Pares (X, X') usados en el cálculo de residuos por fila: (x[i,j-1], x[i,j])."""
    return img[:, :-1].ravel(), img[:, 1:].ravel()

In [ ]:
def entropia_conjunta(a, b):
    """H(A,B) en bits desde el histograma conjunto de dos arrays enteros."""
    a = np.asarray(a); b = np.asarray(b)
    a = a - a.min(); b = b - b.min()
    h = np.zeros((a.max() + 1, b.max() + 1))
    np.add.at(h, (a, b), 1)
    return entropia(h.ravel() / h.sum())


def info_mutua(a, b):
    """I(A;B) = H(A) + H(B) - H(A,B), en bits."""
    return entropia(pmf(a)) + entropia(pmf(b)) - entropia_conjunta(a, b)

### Tests de la base (correr antes de usar)

In [ ]:
# Round-trip de Huffman y de residuos sobre todas las imágenes
for nombre in IMAGENES:
    img = cargar_imagen(nombre)
    # Huffman sobre píxeles
    p = pmf(img); cod = huffman_code(p)
    bits = codificar(img, cod)
    dec = np.array(huffman_decode(bits, cod)).reshape(img.shape)
    assert np.array_equal(dec, img), 'falla Huffman pixel a pixel'
    H, L = entropia(p), longitud_media(p, cod)
    assert H <= L < H + 1, 'se debe cumplir H <= L < H+1'
    assert abs(L - bits_promedio(bits, img.size)) < 1e-9   # L teórica == real (misma PMF)
    # Residuos por filas
    c0, r = residuos_filas(img)
    assert np.array_equal(reconstruir_filas(c0, r), img)
    # Residuos serpentina
    x0, rs = residuos_serpentina(img)
    assert np.array_equal(reconstruir_serpentina(x0, rs, img.shape), img)
    # I(X;X') >= 0 y <= H(X)
    X, Xp = pares_vecinos(img)
    I = info_mutua(X, Xp)
    assert -1e-9 <= I <= entropia(pmf(X)) + 1e-9
    print(f'{nombre}: {img.shape}  H={H:.3f}  L={L:.3f}  I(X;X\')={I:.3f}  OK')

## 1. Ejercicio 1 - píxel a píxel  
**(a)** `p(x)`, gráfico y `H(X)` para cada imagen; comparar entropías.
**(b)** Huffman, bits promedio reales vs `L` teórica (y vs `H(X)` y vs 8 bits).

In [ ]:
import os
os.makedirs('figuras', exist_ok=True)
imgs = {n: cargar_imagen(n) for n in IMAGENES}
pmf_x = {n: pmf(im) for n, im in imgs.items()}
H_x = {n: entropia(p) for n, p in pmf_x.items()}


def grilla(n, ncols=3, w=4.5, h=3.5):
    """Grilla de subplots para n gráficos (los sobrantes quedan apagados)."""
    nrows = -(-n // ncols)
    fig, axs = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows))
    axs = np.atleast_1d(axs).ravel()
    for ax in axs[n:]:
        ax.axis('off')
    return fig, axs


# Imágenes
fig, axs = plt.subplots(1, len(imgs), figsize=(3.2 * len(imgs), 2.6))
for ax, (n, im) in zip(axs, imgs.items()):
    ax.imshow(im, cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'{n} ({im.shape[1]}x{im.shape[0]})', fontsize=9); ax.axis('off')
plt.tight_layout(); plt.savefig('figuras/ej1_imagenes.png', dpi=150); plt.show()

# PMF de cada imagen
fig, axs = grilla(len(imgs))
xs = np.arange(256)
for ax, n in zip(axs, imgs):
    ax.bar(xs, [pmf_x[n].get(x, 0) for x in xs], width=1.0)
    ax.set_xlabel('intensidad x'); ax.set_ylabel('p(x)')
    ax.set_title(f'{n}: H(X) = {H_x[n]:.3f} bits')
plt.tight_layout(); plt.savefig('figuras/ej1_pmf.png', dpi=150); plt.show()
for n in imgs: print(f'{n}: H(X) = {H_x[n]:.4f} bits')

In [ ]:
cod_x, L_x, real_x, bits_x = {}, {}, {}, {}
for n, im in imgs.items():
    cod_x[n] = huffman_code(pmf_x[n])
    L_x[n] = longitud_media(pmf_x[n], cod_x[n])
    bits_x[n] = codificar(im, cod_x[n])
    real_x[n] = bits_promedio(bits_x[n], im.size)
    # L teórica == bits reales (misma PMF usada para codificar); la tabla del código no se cuenta
print(f"{'imagen':10s} {'H(X)':>7s} {'L':>7s} {'reales':>7s} {'ratio 8/bits':>12s}")
for n in imgs:
    print(f'{n:10s} {H_x[n]:7.3f} {L_x[n]:7.3f} {real_x[n]:7.3f} {8/real_x[n]:12.3f}')

## 2. Ejercicio 2 - residuos e información mutua 
**(a)** `p(y)` y `H(Y)`; graficar `p(x)` y `p(y)` juntas.
**(b)** Huffman sobre residuos; bits promedio reales vs `L`. Ojo: sumar el costo de los píxeles iniciales (y de la tabla si se quiere ser estricto).
**(c)** `I(X;X')` entre vecinos.

In [ ]:
col0, res_y, pmf_y, H_y = {}, {}, {}, {}
for n, im in imgs.items():
    col0[n], res_y[n] = residuos_filas(im)
    pmf_y[n] = pmf(res_y[n])
    H_y[n] = entropia(pmf_y[n])

fig, axs = grilla(len(imgs))
xs = np.arange(256); ys = np.arange(-255, 256)
for ax, n in zip(axs, imgs):
    ax.plot(xs, [pmf_x[n].get(x, 0) for x in xs], label=f'p(x)  H={H_x[n]:.2f}')
    ax.plot(ys, [pmf_y[n].get(y, 0) for y in ys], label=f'p(y)  H={H_y[n]:.2f}')
    ax.set_title(n); ax.set_xlabel('valor'); ax.legend()
plt.tight_layout(); plt.savefig('figuras/ej2_pmf.png', dpi=150); plt.show()
for n in imgs: print(f'{n}: H(X) = {H_x[n]:.4f}   H(Y) = {H_y[n]:.4f}   dif = {H_x[n]-H_y[n]:.4f}')

In [ ]:
L_y, real_y, total_y = {}, {}, {}
for n, im in imgs.items():
    cod = huffman_code(pmf_y[n])
    L_y[n] = longitud_media(pmf_y[n], cod)
    bits = codificar(res_y[n], cod)
    # verificación de la cadena completa: decodificar y reconstruir
    dec = np.array(huffman_decode(bits, cod)).reshape(res_y[n].shape)
    assert np.array_equal(reconstruir_filas(col0[n], dec), im)
    real_y[n] = bits_promedio(bits, res_y[n].size)
    # costo total con los M píxeles iniciales (8 bits c/u), por píxel de la imagen
    total_y[n] = (len(bits) + 8 * im.shape[0]) / im.size
print(f"{'imagen':10s} {'H(Y)':>7s} {'L_y':>7s} {'reales':>7s} {'total c/ iniciales':>19s} {'ratio':>7s}")
for n in imgs:
    print(f'{n:10s} {H_y[n]:7.3f} {L_y[n]:7.3f} {real_y[n]:7.3f} {total_y[n]:19.3f} {8/total_y[n]:7.3f}')

In [ ]:
I_xx = {}
for n, im in imgs.items():
    X, Xp = pares_vecinos(im)
    I_xx[n] = info_mutua(X, Xp)
print(f"{'imagen':10s} {'I(X;X*)':>8s} {'H(X)-H(Y)':>10s}")
for n in imgs:
    print(f'{n:10s} {I_xx[n]:8.3f} {H_x[n]-H_y[n]:10.3f}')

## 4. Ejercicio 4 (opcional) - compresor/descompresor `.tp2`
Los programas de línea de comandos son `comprimir_tp2.py` y `descomprimir_tp2.py` (formato en `formato_tp2.py`):
```
python comprimir_tp2.py imagen.jpg              # genera imagen.tp2
python descomprimir_tp2.py imagen.tp2 [-o x.png]  # muestra la imagen
```
Acá se usan las mismas funciones para mostrar el resultado sobre las imágenes del TP.

In [ ]:
from formato_tp2 import comprimir, descomprimir

fig, axs = plt.subplots(2, len(imgs), figsize=(4 * len(imgs), 5))
print(f"{'imagen':10s} {'bytes .tp2':>11s} {'bits/px':>8s} {'ratio':>6s} {'exacta':>7s}")
for k, (n, im) in enumerate(imgs.items()):
    datos = comprimir(im)                               # bytes del archivo .tp2
    rec = descomprimir(datos)
    exacta = np.array_equal(rec, im)
    assert exacta, 'la descompresión no es exacta'
    bpp = 8 * len(datos) / im.size
    print(f'{n:10s} {len(datos):11d} {bpp:8.3f} {8 / bpp:6.2f} {str(exacta):>7s}')
    axs[0, k].imshow(im, cmap='gray', vmin=0, vmax=255); axs[0, k].set_title(f'{n} original'); axs[0, k].axis('off')
    axs[1, k].imshow(rec, cmap='gray', vmin=0, vmax=255); axs[1, k].set_title(f'{n} desde .tp2 ({len(datos)} bytes)'); axs[1, k].axis('off')
plt.tight_layout(); plt.show()